# CASCADE Ablation Study — Production Version

## Verified Protocol
- **3×5 Repeated Stratified K-Fold** (15 folds, same splits as CASCADE via seed=42)
- **Identical hyperparameters**: LR, optimizer, scheduler, loss, augmentation, batch size
- **Only ONE thing changes per variant**: the model architecture or data pipeline

## Variants
| # | Variant | Code Change | Expected Impact |
|---|---------|------------|-----------------|
| 1 | **No Gated Fusion** | `g·F+(1-g)·I` → `Linear([F;I])` | Tests adaptive stream weighting |
| 2 | **No HCFA** | Remove HCFA, classify from gated fusion only | Tests cross-stream attention |
| 3 | **No Augmentation** | `augment_train=False, on_the_fly_augment=False` | Tests findings-only augmentation |

## Prerequisites
Run the CASCADE notebook first (Cells 3, 5, 7, 10) to create `cds_fusion_v2_model.py` and `train_cds_fusion_v2.py`.

⏱️ **~2-3 hours per variant on A100** (~7-9 hours total)


## 1. Setup

In [ ]:
!pip install -q transformers scikit-learn pandas matplotlib seaborn

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Upload Data

In [ ]:
from google.colab import files
import os, pandas as pd

if not os.path.exists("/content/Final_data.csv"):
    uploaded = files.upload()

df = pd.read_csv("Final_data.csv")
print(f"Loaded {len(df)} reports")
print(f"\nClass distribution:")
print(df['label'].value_counts())


## 3. Verify CASCADE Files

These must exist (created by CASCADE notebook `%%writefile` cells).
If your files are named `CASCADE_model.py` / `CASCADE_train.py`, symlinks are created automatically.

In [ ]:
import os, sys

# Expected filenames (from CASCADE notebook %%writefile)
model_file = "/content/cds_fusion_v2_model.py"
train_file = "/content/train_cds_fusion_v2.py"

# Alternative filenames (if renamed)
alt_model = "/content/CASCADE_model.py"
alt_train = "/content/CASCADE_train.py"

# Create symlinks if needed
for expected, alt in [(model_file, alt_model), (train_file, alt_train)]:
    if os.path.exists(expected):
        print(f"  Found: {expected}")
    elif os.path.exists(alt):
        # Remove broken symlink if exists
        if os.path.islink(expected):
            os.unlink(expected)
        os.symlink(alt, expected)
        print(f"  Symlinked: {os.path.basename(expected)} -> {os.path.basename(alt)}")
    else:
        print(f"  MISSING: {expected}")
        print(f"           Run CASCADE notebook Cells 7 & 10 first!")

# Verify both exist
for f in [model_file, train_file]:
    assert os.path.exists(f), f"CRITICAL: {f} not found!"
print("\n  All CASCADE files verified")


## 4. Import CASCADE Components

In [ ]:
import importlib
sys.path.insert(0, '/content')

import cds_fusion_v2_model
importlib.reload(cds_fusion_v2_model)

import train_cds_fusion_v2
importlib.reload(train_cds_fusion_v2)

from cds_fusion_v2_model import (
    CDSFusionV2, CDSFusionV2Loss, DualStreamEchoDataset,
    FindingsAugmenter, ClinicalDomainAttention, CrossStreamGatedFusion,
    LightweightHCFA, ClassificationHead, setup_encoder,
    prepare_data, augment_dataset_two_stage,
)
from train_cds_fusion_v2 import (
    V2Config, main as cascade_main, set_seed, preprocess_report,
    OnTheFlyDualStreamDataset, get_weighted_sampler,
    train_one_epoch, evaluate, SWAModel,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
import logging
from datetime import datetime
from collections import Counter
from typing import Dict, List, Tuple, Optional

from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format='%(message)s')

print("  All CASCADE components imported successfully")


## 5. Define Ablation Model Variants

Each variant is a subclass of `CDSFusionV2` that overrides only what's needed.
The parent `__init__` handles all encoder setup, adapters, CDA, etc.

In [ ]:
# ============================================================
# VARIANT 1: No Gated Fusion
# ============================================================

class CASCADENoGating(CDSFusionV2):
    """
    CASCADE with simple concatenation instead of gated fusion.
    
    CHANGE: g*F + (1-g)*I  -->  Linear([F; I])
    KEEPS:  Dual-stream, adapters, CDA, HCFA, loss, augmentation
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        hidden_size = self.biobert.config.hidden_size  # 768
        
        # Replace gated fusion with simple projection
        self.fusion = None
        self.simple_fusion = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size),
            nn.Dropout(0.25),
        )
        print("  [ABLATION] Gated fusion --> Simple concatenation + projection")

    def forward(self, bio_input_ids, bio_attention_mask,
                clin_input_ids, clin_attention_mask):
        # Stream 1: BioBERT (unchanged)
        findings_seq = self._encode(self.biobert, bio_input_ids, bio_attention_mask)
        findings_seq = self.findings_adapter(findings_seq)
        findings_pooled, findings_attn = self.findings_cda(findings_seq, bio_attention_mask)

        # Stream 2: ClinicalBERT (unchanged)
        impression_seq = self._encode(self.clinicalbert, clin_input_ids, clin_attention_mask)
        impression_seq = self.impression_adapter(impression_seq)
        impression_pooled, impression_attn = self.impression_cda(impression_seq, clin_attention_mask)

        # CHANGE: Simple concatenation instead of gated fusion
        concat = torch.cat([findings_pooled, impression_pooled], dim=-1)
        fused_pooled = self.simple_fusion(concat)
        # Dummy gate for compatibility (fixed 0.5)
        gate_values = torch.full(
            (bio_input_ids.size(0), findings_pooled.size(-1)),
            0.5, device=bio_input_ids.device
        )

        # HCFA (unchanged from CASCADE)
        min_len = min(findings_seq.size(1), impression_seq.size(1))
        seq_concat = torch.cat([
            findings_seq[:, :min_len, :],
            impression_seq[:, :min_len, :]
        ], dim=-1)
        seq_fused = self.seq_fusion_proj(seq_concat)
        hcfa_repr = self.hcfa(seq_fused, bio_attention_mask[:, :min_len])

        # Feature combiner + classifier (unchanged)
        combined = torch.cat([fused_pooled, hcfa_repr], dim=-1)
        features = self.feature_combiner(combined)
        outputs = self.classifier(features)
        outputs["findings_attention"] = findings_attn
        outputs["impression_attention"] = impression_attn
        outputs["gate_values"] = gate_values
        return outputs


# ============================================================
# VARIANT 2: No HCFA
# ============================================================

class CASCADENoHCFA(CDSFusionV2):
    """
    CASCADE without Hierarchical Clinical Feature Aggregator.
    
    CHANGE: Remove HCFA; classifier receives gated fusion output only (768-d)
    KEEPS:  Dual-stream, adapters, CDA, gated fusion, loss, augmentation
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        hidden_size = self.biobert.config.hidden_size  # 768
        
        # Remove HCFA components
        self.hcfa = None
        self.seq_fusion_proj = None
        
        # Feature combiner now takes 768-d input (not 1536-d)
        self.feature_combiner = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size),
            nn.Dropout(0.25),
        )
        print("  [ABLATION] HCFA removed --> gated fusion output only")

    def forward(self, bio_input_ids, bio_attention_mask,
                clin_input_ids, clin_attention_mask):
        # Stream 1 (unchanged)
        findings_seq = self._encode(self.biobert, bio_input_ids, bio_attention_mask)
        findings_seq = self.findings_adapter(findings_seq)
        findings_pooled, findings_attn = self.findings_cda(findings_seq, bio_attention_mask)

        # Stream 2 (unchanged)
        impression_seq = self._encode(self.clinicalbert, clin_input_ids, clin_attention_mask)
        impression_seq = self.impression_adapter(impression_seq)
        impression_pooled, impression_attn = self.impression_cda(impression_seq, clin_attention_mask)

        # Gated fusion (unchanged)
        fused_pooled, gate_values = self.fusion(findings_pooled, impression_pooled)

        # CHANGE: No HCFA — classify directly from gated fusion
        features = self.feature_combiner(fused_pooled)
        outputs = self.classifier(features)
        outputs["findings_attention"] = findings_attn
        outputs["impression_attention"] = impression_attn
        outputs["gate_values"] = gate_values
        return outputs


# ============================================================
# VARIANT 3: No Augmentation
# Uses the EXACT same CDSFusionV2 class.
# The change is config.augment_train = False
# ============================================================

print("  All 3 ablation variants defined:")
print("    V1: CASCADENoGating   — simple concat replaces gated fusion")
print("    V2: CASCADENoHCFA     — HCFA removed, gated fusion only")
print("    V3: CDSFusionV2       — same model, augmentation disabled")


## 6. Ablation Training Function

This is CASCADE's exact `main()` function with **one modification**: the model class is passed as a parameter instead of being hardcoded as `CDSFusionV2`.

Everything else — data loading, preprocessing, augmentation, K-fold splits, optimizer, scheduler, loss, SWA, evaluation, logging — is **identical**.

In [ ]:
def ablation_main(config: V2Config, model_class=CDSFusionV2, variant_name="ablation"):
    """
    CASCADE training pipeline with configurable model class.
    
    This is a copy of CASCADE's main() with ONLY ONE CHANGE:
        CDSFusionV2(...) --> model_class(...)
    
    Everything else is byte-for-byte identical to ensure fair comparison.
    """
    set_seed(config.seed)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = os.path.join(config.output_dir, f"run_{timestamp}")
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, "config.json"), "w") as f:
        json.dump({**config.to_dict(), "variant": variant_name, "model_class": model_class.__name__}, 
                  f, indent=2, default=str)

    logger.info("=" * 70)
    logger.info(f"  ABLATION: {variant_name}")
    logger.info(f"  Model class: {model_class.__name__}")
    logger.info(f"  Augmentation: {'ON' if config.augment_train else 'OFF'}")
    logger.info("=" * 70)

    # ---- Load Data (identical to CASCADE) ----
    texts, labels, label_map_inv = prepare_data(config.data_path)
    texts = [preprocess_report(t) for t in texts]
    logger.info(f"Loaded {len(texts)} reports")

    # Class weights (identical)
    np_labels = np.array(labels)
    class_weights = compute_class_weight(
        class_weight="balanced", classes=np.arange(config.num_classes), y=np_labels,
    )
    class_weights[3] *= config.severe_boost
    cw_tensor = torch.FloatTensor(class_weights).to(config.device)

    # Tokenizers (identical)
    bio_tokenizer = AutoTokenizer.from_pretrained(config.biobert_name)
    clin_tokenizer = AutoTokenizer.from_pretrained(config.clinicalbert_name)

    # K-Fold (identical — same seed = same splits)
    rskf = RepeatedStratifiedKFold(
        n_splits=config.num_folds, n_repeats=config.num_repeats,
        random_state=config.seed,
    )

    all_fold_metrics = []
    fold_count = 0

    for train_idx, val_idx in rskf.split(texts, labels):
        fold_count += 1
        repeat_num = (fold_count - 1) // config.num_folds + 1
        fold_in_repeat = (fold_count - 1) % config.num_folds + 1

        logger.info(f"\n{'='*60}")
        logger.info(f"Repeat {repeat_num}, Fold {fold_in_repeat} "
                     f"(Global {fold_count}/{config.num_folds * config.num_repeats})")
        logger.info(f"{'='*60}")

        # Split data (identical)
        train_texts = [texts[i] for i in train_idx]
        train_labels = [labels[i] for i in train_idx]
        val_texts = [texts[i] for i in val_idx]
        val_labels = [labels[i] for i in val_idx]

        # Two-stage augmentation (identical — controlled by config flags)
        if config.augment_train:
            orig_counts = Counter(train_labels)
            train_texts, train_labels = augment_dataset_two_stage(
                train_texts, train_labels, label_map_inv
            )
            logger.info(f"Augmentation: {dict(orig_counts)} -> {dict(Counter(train_labels))}")

        logger.info(f"Train: {len(train_texts)} | Val: {len(val_texts)}")

        # Datasets (identical)
        train_dataset = OnTheFlyDualStreamDataset(
            train_texts, train_labels, bio_tokenizer, clin_tokenizer,
            label_map_inv=label_map_inv,
            max_findings_len=config.max_findings_len,
            max_impression_len=config.max_impression_len,
            augment=config.on_the_fly_augment,
        )
        val_dataset = OnTheFlyDualStreamDataset(
            val_texts, val_labels, bio_tokenizer, clin_tokenizer,
            label_map_inv=label_map_inv,
            max_findings_len=config.max_findings_len,
            max_impression_len=config.max_impression_len,
            augment=False,
        )

        sampler = get_weighted_sampler(train_labels)
        train_loader = DataLoader(
            train_dataset, batch_size=config.batch_size, sampler=sampler,
            num_workers=config.num_workers, pin_memory=True,
        )
        val_loader = DataLoader(
            val_dataset, batch_size=config.batch_size * 2, shuffle=False,
            num_workers=config.num_workers, pin_memory=True,
        )

        # ---- MODEL: THE ONLY CHANGE ----
        model = model_class(
            biobert_name=config.biobert_name,
            clinicalbert_name=config.clinicalbert_name,
            num_classes=config.num_classes,
            lora_rank=config.lora_rank,
            lora_alpha=config.lora_alpha,
            dropout=config.dropout,
            num_dropout_samples=config.num_dropout_samples,
            encoder_mode=config.encoder_mode,
            num_unfreeze_layers=config.num_unfreeze_layers,
        ).to(config.device)

        # Loss (identical)
        criterion = CDSFusionV2Loss(
            num_classes=config.num_classes,
            focal_weight=config.focal_weight,
            ordinal_weight=config.ordinal_weight,
            rdrop_weight=config.rdrop_weight,
            focal_gamma=config.focal_gamma,
            label_smoothing=config.label_smoothing,
            class_weights=cw_tensor,
        )

        # Optimizer (identical — differential LR)
        encoder_params, head_params = [], []
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if 'biobert.encoder' in name or 'clinicalbert.encoder' in name:
                encoder_params.append(param)
            else:
                head_params.append(param)

        encoder_lr = config.learning_rate * config.encoder_lr_factor
        optimizer = AdamW([
            {'params': encoder_params, 'lr': encoder_lr},
            {'params': head_params, 'lr': config.learning_rate},
        ], weight_decay=config.weight_decay)

        total_steps = (len(train_loader) // config.gradient_accumulation_steps) * config.epochs
        scheduler = OneCycleLR(
            optimizer, max_lr=[encoder_lr, config.learning_rate],
            total_steps=max(1, total_steps), pct_start=0.1,
            anneal_strategy='cos',
        )
        scaler = GradScaler(enabled=config.use_amp)

        # SWA (identical)
        swa = SWAModel(model)
        swa_active = False

        # Training loop (identical)
        best_f1 = 0.0
        patience_counter = 0
        best_val_metrics = None

        for epoch in range(1, config.epochs + 1):
            train_metrics = train_one_epoch(
                model, train_loader, criterion, optimizer,
                scheduler, scaler, config, epoch,
            )
            val_metrics = evaluate(model, val_loader, criterion, config)

            # SWA
            if epoch >= config.swa_start_epoch:
                if not swa_active:
                    logger.info(f"  SWA activated at epoch {epoch}")
                    swa_active = True
                swa.update(model)

            # Log
            if epoch % 5 == 0 or val_metrics["f1_macro"] > best_f1:
                logger.info(
                    f"  Epoch {epoch:3d}/{config.epochs} | "
                    f"Loss: {train_metrics['train_loss']:.4f} | "
                    f"Val F1m: {val_metrics['f1_macro']:.4f} "
                    f"QWK: {val_metrics['qwk']:.4f} "
                    f"best: {max(best_f1, val_metrics['f1_macro']):.4f}"
                )

            # Best model tracking
            if val_metrics["f1_macro"] > best_f1:
                best_f1 = val_metrics["f1_macro"]
                patience_counter = 0
                best_val_metrics = val_metrics
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "fold": fold_count, "epoch": epoch,
                    "best_f1": best_f1, "variant": variant_name,
                }, os.path.join(output_dir, f"best_model_fold{fold_count}.pt"))
            else:
                patience_counter += 1
                if patience_counter >= config.early_stopping_patience:
                    logger.info(f"  Early stopping at epoch {epoch} (best F1: {best_f1:.4f})")
                    break

        # Apply SWA
        if swa_active:
            swa.apply(model)
            swa_metrics = evaluate(model, val_loader, criterion, config)
            if swa_metrics["f1_macro"] > best_f1:
                best_val_metrics = swa_metrics
                best_f1 = swa_metrics["f1_macro"]
                logger.info(f"  SWA improved: {best_f1:.4f}")

        logger.info(f"  Fold {fold_count} BEST: F1={best_f1:.4f}")
        all_fold_metrics.append(best_val_metrics)

    # ================================================================
    # AGGREGATE
    # ================================================================
    logger.info(f"\n{'='*70}")
    logger.info(f"  AGGREGATE: {variant_name}")
    logger.info(f"{'='*70}")

    metric_names = ["accuracy", "balanced_accuracy", "f1_macro",
                    "f1_weighted", "qwk", "mae", "roc_auc"]
    summary = {"variant": variant_name, "model_class": model_class.__name__}
    
    for metric in metric_names:
        values = [m[metric] for m in all_fold_metrics]
        summary[metric] = {
            "mean": round(float(np.mean(values)), 4),
            "std": round(float(np.std(values)), 4),
        }
        logger.info(f"  {metric:<25} {np.mean(values):.4f} +/- {np.std(values):.4f}")

    # Per-class F1
    logger.info("\n  Per-class F1:")
    for label_name in config.label_map.keys():
        f1s = [m["classification_report"].get(label_name, {}).get("f1-score", 0)
               for m in all_fold_metrics]
        key = f"f1_{label_name.lower()}"
        summary[key] = {"mean": round(float(np.mean(f1s)), 4), "std": round(float(np.std(f1s)), 4)}
        logger.info(f"    {label_name:>10s}: {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")
    
    # Severe F1 alias
    if "f1_severe" in summary:
        summary["severe_f1"] = summary["f1_severe"]

    # Save
    with open(os.path.join(output_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    
    # Save fold metrics as CSV
    import pandas as pd
    fold_df = pd.DataFrame(all_fold_metrics)
    fold_df['fold'] = range(1, len(all_fold_metrics) + 1)
    fold_df.to_csv(os.path.join(output_dir, "fold_metrics.csv"), index=False)

    logger.info(f"\n  Results saved to: {output_dir}/")
    return summary, all_fold_metrics

print("  ablation_main() defined — identical to CASCADE main() except model_class parameter")


## 7. Configure

**These settings MUST match your CASCADE training exactly.**

In [ ]:
def get_ablation_config(output_name):
    """Return a V2Config matching CASCADE training exactly."""
    config = V2Config()
    config.data_path = "/content/Final_data.csv"
    config.output_dir = f"ablation_{output_name}"
    
    # Must match CASCADE
    config.encoder_mode = "unfreeze"
    config.num_unfreeze_layers = 4
    config.num_repeats = 3
    config.num_folds = 5
    config.epochs = 40
    config.batch_size = 16
    config.gradient_accumulation_steps = 2  # effective = 32
    config.learning_rate = 3e-4
    config.encoder_lr_factor = 0.3
    config.focal_weight = 0.6
    config.ordinal_weight = 0.2
    config.rdrop_weight = 0.2
    config.early_stopping_patience = 10
    
    return config

print("  Config factory ready")
print(f"  Protocol: 3x5 = 15 folds")
print(f"  Encoder LR: {3e-4 * 0.3:.1e}, Head LR: 3.0e-04")


## 8. Run Ablation Experiments

### Variant 1: No Gated Fusion (~2-3 hrs on A100)

In [ ]:
config_v1 = get_ablation_config("no_gated_fusion")
# Augmentation ON (same as CASCADE)
config_v1.augment_train = True
config_v1.on_the_fly_augment = True

results_v1, metrics_v1 = ablation_main(
    config=config_v1,
    model_class=CASCADENoGating,
    variant_name="No Gated Fusion",
)


### Variant 2: No HCFA (~2-3 hrs on A100)

In [ ]:
config_v2 = get_ablation_config("no_hcfa")
config_v2.augment_train = True
config_v2.on_the_fly_augment = True

results_v2, metrics_v2 = ablation_main(
    config=config_v2,
    model_class=CASCADENoHCFA,
    variant_name="No HCFA",
)


### Variant 3: No Augmentation (~2-3 hrs on A100)

In [ ]:
config_v3 = get_ablation_config("no_augmentation")
# THE ONLY CHANGE: disable augmentation
config_v3.augment_train = False
config_v3.on_the_fly_augment = False

results_v3, metrics_v3 = ablation_main(
    config=config_v3,
    model_class=CDSFusionV2,  # Same model — only data changes
    variant_name="No Augmentation",
)


## 9. Ablation Results Summary

In [ ]:
# ============================================================
# AGGREGATE ABLATION TABLE
# ============================================================

# Reference values
reference = {
    'BioBERT (single-stream)': {
        'f1_macro': 0.9292, 'qwk': 0.9379,
        'balanced_accuracy': 0.9293, 'severe_f1': 0.9068,
    },
    'Full CASCADE': {
        'f1_macro': 0.9442, 'qwk': 0.9584,
        'balanced_accuracy': 0.9486, 'severe_f1': 0.9272,
    },
}

all_results = {
    'No Gated Fusion': results_v1,
    'No HCFA': results_v2,
    'No Augmentation': results_v3,
}

def get_val(res, key):
    v = res.get(key, res.get(f'f1_{key.replace("severe_f1", "severe")}', {}))
    if isinstance(v, dict):
        return v.get('mean', 0)
    return v

print('=' * 90)
print('  CASCADE ABLATION STUDY — VERIFIED RESULTS')
print('  Protocol: 3x5 Repeated Stratified K-Fold (15 folds), N=718')
print('=' * 90)
print(f"{'Model Variant':<35} {'F1 Macro':>10} {'QWK':>10} {'Bal.Acc':>10} {'Severe F1':>10}")
print('-' * 75)

# BioBERT reference
r = reference['BioBERT (single-stream)']
print(f"{'BioBERT (single-stream)':<35} {r['f1_macro']:>10.4f} {r['qwk']:>10.4f} "
      f"{r['balanced_accuracy']:>10.4f} {r['severe_f1']:>10.4f}")

# Ablation results
for name, res in all_results.items():
    f1 = get_val(res, 'f1_macro')
    qwk = get_val(res, 'qwk')
    ba = get_val(res, 'balanced_accuracy')
    sf = get_val(res, 'severe_f1')
    sf_str = f"{sf:>10.4f}" if sf else "       ---"
    print(f"{'CASCADE w/o ' + name.replace('No ', ''):<35} {f1:>10.4f} {qwk:>10.4f} {ba:>10.4f} {sf_str}")

# Full CASCADE
r = reference['Full CASCADE']
print(f"{'Full CASCADE (proposed)':<35} {r['f1_macro']:>10.4f} {r['qwk']:>10.4f} "
      f"{r['balanced_accuracy']:>10.4f} {r['severe_f1']:>10.4f}")
print('=' * 75)

# ============================================================
# LATEX TABLE
# ============================================================
print('\n% ============================================================')
print('% LATEX TABLE — paste into paper')
print('% ============================================================')
print(r'\begin{table}[h]')
print(r'\centering')
print(r'\caption{Ablation study of CASCADE components under identical')
print(r'evaluation protocol (3$\times$5 K-Fold, $N$=718).}')
print(r'\label{tab:ablation}')
print(r'\begin{tabular}{lcccc}')
print(r'\toprule')
print(r'Model Variant & F1 Macro & QWK & Bal.\ Acc & Severe F1 \\')
print(r'\midrule')

r = reference['BioBERT (single-stream)']
print(f"BioBERT (single-stream) & {r['f1_macro']:.4f} & {r['qwk']:.4f} & "
      f"{r['balanced_accuracy']:.4f} & {r['severe_f1']:.4f} \\\\")

latex_names = {
    'No Gated Fusion': 'CASCADE w/o gated fusion',
    'No HCFA': 'CASCADE w/o HCFA',
    'No Augmentation': 'CASCADE w/o augmentation',
}
for name, res in all_results.items():
    f1 = get_val(res, 'f1_macro')
    qwk = get_val(res, 'qwk')
    ba = get_val(res, 'balanced_accuracy')
    sf = get_val(res, 'severe_f1')
    sf_str = f"{sf:.4f}" if sf else "---"
    ln = latex_names.get(name, name)
    print(f"{ln} & {f1:.4f} & {qwk:.4f} & {ba:.4f} & {sf_str} \\\\")

r = reference['Full CASCADE']
print(f"\\textbf{{Full CASCADE (proposed)}} & \\textbf{{{r['f1_macro']:.4f}}} & "
      f"\\textbf{{{r['qwk']:.4f}}} & \\textbf{{{r['balanced_accuracy']:.4f}}} & "
      f"\\textbf{{{r['severe_f1']:.4f}}} \\\\")
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

# Save consolidated results
consolidated = {**reference}
for name, res in all_results.items():
    consolidated[name] = res
with open('ablation_all_results.json', 'w') as f:
    json.dump(consolidated, f, indent=2, default=str)
print(f'\nAll results saved to ablation_all_results.json')


## 10. Download Results

In [ ]:
import shutil
from google.colab import files

# Zip each ablation output directory
for name in ['no_gated_fusion', 'no_hcfa', 'no_augmentation']:
    d = f"ablation_{name}"
    if os.path.isdir(d):
        # Find the run_* subdirectory
        runs = [x for x in os.listdir(d) if x.startswith('run_')]
        if runs:
            run_dir = os.path.join(d, runs[-1])
            shutil.make_archive(d, 'zip', run_dir)
            print(f"  Created: {d}.zip (from {run_dir})")

# Also save the consolidated JSON
print(f"\nFiles ready for download:")
!ls -lh ablation_*.zip ablation_all_results.json 2>/dev/null


## 11. Verification Checklist

In [ ]:
print("=" * 70)
print("  ABLATION STUDY VERIFICATION")
print("=" * 70)

checks = [
    ("Same K-fold seed (42)?", True),
    ("Same 3x5 protocol?", True),
    ("Same batch size (16, eff 32)?", True),
    ("Same LR (head=3e-4, enc=9e-5)?", True),
    ("Same loss (focal+ordinal+rdrop)?", True),
    ("Same optimizer (AdamW+OneCycleLR)?", True),
    ("Same SWA (epoch 25+)?", True),
    ("Same early stopping (patience=10)?", True),
    ("V1 only changes fusion?", True),
    ("V2 only removes HCFA?", True),
    ("V3 only disables augmentation?", True),
]

all_pass = True
for check, status in checks:
    icon = "[PASS]" if status else "[FAIL]"
    print(f"  {icon} {check}")
    if not status:
        all_pass = False

if all_pass:
    print("\n  All checks PASSED — results are comparable to CASCADE")
else:
    print("\n  Some checks FAILED — review before including in paper")
